# RedZone world model: ladder 18m + 60m on free Kaggle GPU T4x2 (EXP-107g)

Personal use only: plan B data, so every grades.json carries "personal-use, not for commercial release".
The TPU ladder waits on Kaggle's one-time identity check; this runs its two low rungs on the free GPU quota instead
($0, no Colab CU). One point per T4, in parallel: 18m on GPU 0, 60m on GPU 1, each on the same 0.6B-token budget,
fp16 autocast with a grad scaler (T4 has no bf16), base recipe (AdamW, cosine). Leagues NBA + NFL (NBA first).
Data: public play by play from GitHub (NBA.com cdn NBA, nflverse NFL), tokenized with vocab v2. Locked and
scorecard nights are never downloaded.

Settings > Accelerator > **GPU T4 x2**, Internet **on**, then **Save Version > Save & Run All (Commit)**.
About 6 h of the weekly GPU quota (data ~30 min, 18m ~1 h, 60m ~5 h, side by side). Checkpoints go to
/kaggle/working every 30 min: if the session ends, attach this notebook's previous output and run again; finished
points are skipped. The last cell prints the table (NBA first).


In [ ]:
PIN = "8b2ea863d93dd9b47abfa137741cd0713971583a"   # kaggle-inputs commit: factory code pack
REPO = "BouncySlime1215/redzone-everything"
LEAGUES = "nba,nfl"
POINTS = {"18m": 0, "60m": 1}               # preset -> GPU index
MINUTES = {"18m": 240, "60m": 600}         # caps; the token budget ends the runs first
CTX, BATCH, TRAIN_TOKENS = 2048, 16, 6e8
RUN, CODE = "/kaggle/working/rz", "/kaggle/working/rz/code"


In [ ]:
import glob, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
T_START = time.time()
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if torch.cuda.device_count() < 2:
    print("only", torch.cuda.device_count(), "GPU: the points run one after the other")
os.makedirs(CODE, exist_ok=True)
for f in glob.glob("/kaggle/input/**/rz/nfm/*/*", recursive=True):   # resume from an attached previous output
    dst = f"{RUN}/nfm/" + "/".join(f.split("/")[-2:])
    if not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(f, dst)
if not os.path.exists(f"{CODE}/rz/scripts/nfm_train.py"):
    for i, ref in enumerate((PIN, PIN[:12], PIN, PIN[:12])):
        url = f"https://raw.githubusercontent.com/{REPO}/{ref}/factory/rz_factory_code.tar.gz"
        try:
            print("downloading", url); urllib.request.urlretrieve(url, f"{RUN}/code.tar.gz"); break
        except Exception as e:
            print(" ", type(e).__name__, e); time.sleep(30 * i)
    else:
        raise SystemExit("could not download the code pack; run again in a minute")
    with tarfile.open(f"{RUN}/code.tar.gz") as t:
        t.extractall(CODE, filter="data")
# data once, before the two runs start (they share it read-only)
sys.path[:0] = [f"{CODE}/rz", f"{CODE}/rz/scripts"]
import world_model_pilot as W
from pathlib import Path
games = W.build_games(LEAGUES.split(","), Path(f"{RUN}/data"), False, 2)
print("games:", len(games), f"data built in {(time.time() - T_START) / 60:.0f} min")
del games


In [ ]:
def launch(preset, gpu):
    out = f"{RUN}/nfm/{preset}"
    os.makedirs(out, exist_ok=True)
    if os.path.exists(f"{out}/grades.json"):
        print(preset, "done already"); return None
    cmd = [sys.executable, "-u", f"{CODE}/rz/scripts/nfm_train.py", "--out", out, "--ckpt-dir", out,
           "--data-dir", f"{RUN}/data", "--build-vocab", "2", "--leagues", LEAGUES, "--preset", preset,
           "--ctx", str(CTX), "--batch", str(BATCH), "--train-tokens", str(TRAIN_TOKENS), "--device", "cuda",
           "--minutes", str(MINUTES[preset]), "--exp", "EXP-107g"]
    gpu = gpu if torch.cuda.device_count() > 1 else 0
    log = open(f"{out}/train.log", "a")
    return preset, log, subprocess.Popen(cmd, cwd=RUN, stdout=log, stderr=subprocess.STDOUT,
                                         env={**os.environ, "PYTHONUNBUFFERED": "1", "CUDA_VISIBLE_DEVICES": str(gpu)})

procs = []
for preset, gpu in POINTS.items():
    if torch.cuda.device_count() < 2 and procs:
        for _, _, p in procs:
            p.wait()
    r = launch(preset, gpu)
    if r:
        procs.append(r)
while any(p.poll() is None for _, _, p in procs):              # progress every 10 min (last eval line per point)
    time.sleep(600)
    for preset, _, p in procs:
        lines = [l for l in open(f"{RUN}/nfm/{preset}/train.log") if l.startswith("step ")]
        print(f"[{(time.time() - T_START) / 3600:.2f} h] {preset}:", lines[-1].strip() if lines else "starting")
for preset, log, p in procs:
    log.close(); print(preset, "exit code:", p.returncode)


In [ ]:
print("preset | non-emb params | best valid | valid NBA | valid NFL | test next-kind NBA (CI) | tok/s | TFLOPS | amp | license")
for preset in POINTS:
    gp = f"{RUN}/nfm/{preset}/grades.json"
    if not os.path.exists(gp):
        print(preset, "| no grades.json; see", f"{RUN}/nfm/{preset}/train.log"); continue
    g = json.load(open(gp))
    v, t = g.get("valid_loss_by_league", {}), g.get("test_by_league", {}).get("nba", {}).get("next_kind_acc", {})
    print(" | ".join(str(x) for x in (preset, g["params"]["non_embedding_active"], g["best_valid_loss"], v.get("nba"),
                                      v.get("nfl"), f"{t.get('value')} {t.get('ci95')}", g["throughput"]["tokens_per_s"],
                                      g["throughput"]["achieved_tflops"], g["recipe"].get("amp"), g["license"])))
print(f"session wall clock: {(time.time() - T_START) / 3600:.2f} h")
for f in glob.glob(f"{RUN}/nfm/*/ckpt.pt"):     # keep best.pt + grades; the 30-min ckpt only matters mid-run
    if os.path.exists(os.path.join(os.path.dirname(f), "grades.json")):
        os.remove(f)
shutil.rmtree(CODE, ignore_errors=True)
